### Import Packages

In [1]:
import duckdb
from pathlib import Path

#### Configs and Variables

In [2]:
data_vault_path = r"C:\Users\JacksonDivakarRajasi\OneDrive - JMAN Group Ltd\Documents\Python Folder\DuckDB Environment\Olist Eccomerce DuckDB\DuckDB Data\data_vault.duckdb"

#### DuckDB Connection

In [3]:
con = duckdb.connect(data_vault_path)

In [4]:
con.execute("""
CREATE SCHEMA IF NOT EXISTS business_vault;
""")

## Create the As of Table

In [5]:
con.execute("""
CREATE OR REPLACE TABLE business_vault.as_of_date AS

WITH date_range AS (
    SELECT
        DATE_TRUNC('month', MIN(order_purchase_timestamp)) AS min_month,
        DATE_TRUNC('month', MAX(order_purchase_timestamp)) AS max_month
    FROM raw_vault.sat_order_details
)

SELECT
    LAST_DAY(month_date) AS as_of_date

FROM date_range,
GENERATE_SERIES(
    min_month,
    max_month,
    INTERVAL '1 month'
) AS t(month_date);
""")

### Customer PIT

In [6]:
con.execute("""
CREATE OR REPLACE TABLE business_vault.pit_customer AS

SELECT
    h.customer_hk,
    a.as_of_date,

    MAX(s.load_date) AS customer_sat_load_date

FROM raw_vault.hub_customer h

CROSS JOIN business_vault.as_of_date a

LEFT JOIN raw_vault.sat_customer_details s
    ON h.customer_hk = s.customer_hk
   AND s.load_date <= a.as_of_date

GROUP BY
    h.customer_hk,
    a.as_of_date

HAVING MAX(s.load_date) IS NOT NULL;
""")

### Product PIT

In [7]:
con.execute("""
CREATE OR REPLACE TABLE business_vault.pit_product AS

SELECT
    h.product_hk,
    a.as_of_date,
    MAX(s.load_date) AS product_sat_load_date

FROM raw_vault.hub_product h

CROSS JOIN business_vault.as_of_date a

LEFT JOIN raw_vault.sat_product_details s
    ON h.product_hk = s.product_hk
   AND s.load_date <= a.as_of_date

GROUP BY
    h.product_hk,
    a.as_of_date

HAVING MAX(s.load_date) IS NOT NULL;
""")

### Seller PIT

In [8]:
con.execute("""
CREATE OR REPLACE TABLE business_vault.pit_seller AS

SELECT
    h.seller_hk,
    a.as_of_date,
    MAX(s.load_date) AS seller_sat_load_date

FROM raw_vault.hub_seller h

CROSS JOIN business_vault.as_of_date a

LEFT JOIN raw_vault.sat_seller_details s
    ON h.seller_hk = s.seller_hk
   AND s.load_date <= a.as_of_date

GROUP BY
    h.seller_hk,
    a.as_of_date

HAVING MAX(s.load_date) IS NOT NULL;
""")

### Order PIT

In [9]:
con.execute("""
CREATE OR REPLACE TABLE business_vault.pit_order AS

SELECT
    h.order_hk,
    a.as_of_date,
    MAX(s.load_date) AS order_sat_load_date

FROM raw_vault.hub_order h

CROSS JOIN business_vault.as_of_date a

LEFT JOIN raw_vault.sat_order_details s
    ON h.order_hk = s.order_hk
   AND s.load_date <= a.as_of_date

GROUP BY
    h.order_hk,
    a.as_of_date

HAVING MAX(s.load_date) IS NOT NULL;
""")